In [ ]:
# ==========================================================
# セル1：ライブラリの読み込みと NR-500形式リーダの定義
# ==========================================================
# 【対象ファイル】1行目が "#BeginHeader,<行数>" で始まるCSV（KEYENCE NR-500）
#   Futaba形式（1行目が "Time:" で始まる）は 01/02/03 を使ってください。
# ==========================================================
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict

# ---- グラフの日本語文字化け対策（Windows最適化）----
plt.rcParams["font.family"] = ["Meiryo", "Yu Gothic", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*findfont: Font family.*not found.*"
)

# ==========================================================
# ★ 単位換算：圧力[MPa] = 電圧[V] × 20
#   NR-500 は「スケーリング OFF」で生電圧のまま記録されているため換算が必要。
#   係数を変える場合はこの1箇所だけ書き換えれば全体に反映されます。
# ==========================================================
VOLT_TO_MPA = 20.0


def probe_nr500(path):
    """先頭のヘッダだけを読んで NR-500形式か判定し、メタ情報を返す。

    NR-500形式でなければ None を返す（Futaba形式などはここで弾かれる）。
    """
    with open(path, "r", encoding="cp932", errors="replace") as f:
        head = [f.readline().rstrip("\n") for _ in range(120)]

    if not head[0].startswith("#BeginHeader"):
        return None

    # "#BeginHeader,71" → ヘッダ71行。列名行はその71行目なので skiprows=70
    n_header = int(head[0].split(",")[1])

    # "サンプリング周期,5μs" を秒に変換
    # （16行目の "実サンプリング周期" は先頭が「実」なので startswith で誤マッチしない）
    raw = next(l.split(",")[1] for l in head if l.startswith("サンプリング周期"))
    m = re.match(r"\s*([\d.]+)\s*(μs|us|ms|s)\s*$", raw)
    if m is None:
        raise ValueError(f"サンプリング周期を解釈できません: {raw!r}")
    dt = float(m.group(1)) * {"μs": 1e-6, "us": 1e-6, "ms": 1e-3, "s": 1.0}[m.group(2)]

    # "データ数,1000000" → これを nrows に使い、末尾3行のフッタ
    # （#BeginMark,3 / CH名,... / #EndMark）を構造的に除外する
    n_data = int(next(l.split(",")[1] for l in head if l.startswith("データ数")))

    return {"skiprows": n_header - 1, "dt": dt, "n_data": n_data}


def read_nr500(path, channels=("V03", "V04")):
    """NR-500形式CSVを読み、指定チャンネルを MPa に換算した DataFrame と dt を返す。"""
    meta = probe_nr500(path)
    if meta is None:
        raise ValueError("NR-500形式ではありません")

    header = pd.read_csv(
        path, skiprows=meta["skiprows"], encoding="cp932", nrows=0
    ).columns.tolist()

    # "V03" → "(1)HA-V03" を末尾一致で解決（ユニット番号が変わっても追従できる）
    resolved = {}
    for ch in channels:
        hit = [c for c in header if c.endswith(ch)]
        if not hit:
            raise ValueError(f"チャンネル '{ch}' が見つかりません（実際の列: {header}）")
        resolved[ch] = hit[0]

    read_kw = dict(
        skiprows=meta["skiprows"],
        encoding="cp932",
        usecols=list(resolved.values()),
        nrows=meta["n_data"],
    )
    try:
        # dtype を明示しないと pandas 3.0.3 では usecols 使用時に
        # IndexError: list index out of range が出るため、必ず指定する
        df = pd.read_csv(path, dtype="float32", **read_kw)
    except ValueError:
        # 数値以外が混入していた場合の保険
        df = pd.read_csv(path, dtype=str, **read_kw)
        df = df.apply(pd.to_numeric, errors="coerce").dropna().astype("float32")

    df = df.rename(columns={v: k for k, v in resolved.items()})[list(resolved)]
    return df * VOLT_TO_MPA, meta["dt"]


def spectrum(y, dt, f_lo, f_hi):
    """振幅スペクトルを計算し、表示帯域 f_lo〜f_hi だけに切り詰めて返す。

    NR-500 は 100万点あり rfft 後も 500,001点になる。全点を matplotlib に
    渡すと重ね描きで停止するため、★描画前に必ず帯域を切る★。
    0〜150Hz なら Δf=0.2Hz で 751点まで落ちる。
    """
    N = len(y)
    amp = np.abs(np.fft.rfft(y)) / (N / 2)
    amp[0] /= 2  # 直流成分だけは N で割る
    freq = np.fft.rfftfreq(N, d=dt)
    m = (freq >= f_lo) & (freq <= f_hi)
    return freq[m], amp[m]


def folder_tag(dirpath, main_dir):
    """main_dir からの相対パスをファイル名用のタグにする。

    末端フォルダ名だけだと 0.5mm/190℃ と 1.0mm/190℃ が同名になり
    上書きされてしまうため、相対パス全体を使って衝突を防ぐ。
    """
    rel = os.path.relpath(dirpath, main_dir)
    return "root" if rel == "." else rel.replace(os.sep, "_")


print("✅ ライブラリの読み込みと NR-500形式リーダの定義が完了しました。")
print(f"   単位換算: 電圧[V] × {VOLT_TO_MPA} = 圧力[MPa]")

In [ ]:
# ==========================================================
# セル2：メインフォルダの選択
# ==========================================================
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
main_dir = filedialog.askdirectory(title="解析対象のメインフォルダを選択してください")
root.destroy()

if not main_dir:
    print("⚠️ フォルダ選択がキャンセルされました。次のセルには進まず、やり直してください。")
else:
    print(f"✅ 選択されたメインフォルダ:\n{main_dir}")

In [ ]:
# ==========================================================
# セル3：対象ファイルの列挙と NR-500形式の絞り込み
# ==========================================================
if not main_dir:
    raise ValueError("メインフォルダが選択されていません。セル2を再実行してください。")

all_csv = []
for dirpath, dirnames, filenames in os.walk(main_dir):
    for f in filenames:
        if f.lower().endswith(".csv"):
            all_csv.append(os.path.join(dirpath, f))

# NR-500形式だけを対象にする。除外したファイルも件数と例を必ず表示し、
# 「黙って処理されていない」状態が起きないようにする。
target_files, skipped_files = [], []
for p in all_csv:
    try:
        (target_files if probe_nr500(p) is not None else skipped_files).append(p)
    except Exception as e:
        skipped_files.append(p)

print(f"🔍 見つかったCSVファイル: 合計 {len(all_csv)} 件")
print(f"   ├ NR-500形式 : {len(target_files)} 件 ← これを処理します")
print(f"   └ それ以外   : {len(skipped_files)} 件 ← 対象外として除外します")

if target_files:
    print("\n【処理対象ファイルの例】")
    for f in target_files[:3]:
        print(" -", f)
    if len(target_files) > 3:
        print(f"   ... (他 {len(target_files) - 3} 件)")
else:
    print("\n⚠️ NR-500形式のCSVが1件も見つかりませんでした。")
    print("   Futaba形式（1行目が 'Time:'）なら 01/02/03 を使ってください。")

if skipped_files:
    print("\n【対象外として除外したファイルの例】")
    for f in skipped_files[:3]:
        print(" -", os.path.basename(f))
    if len(skipped_files) > 3:
        print(f"   ... (他 {len(skipped_files) - 3} 件)")

In [ ]:
# ==========================================================
# セル4：解析設定とファイルのグループ化（準備）
# ==========================================================
if not target_files:
    raise ValueError("処理対象のNR-500形式ファイルがありません。セル3を確認してください。")

# ==========================================
# ⚙️ 設定：解析したいチャンネルとグラフの表示設定
# ==========================================
target_channels = ["V03", "V04"]   # CH03/CH04 に相当

# --- X軸（Frequency [Hz]）の設定 ---
x_min = 0          # 最小値
x_max = 150        # 最大値（Futaba版03と揃えてあります）
x_step = 10        # 目盛りの間隔

# --- Y軸（Amplitude [MPa]）の設定 ---
# ※ y_max を None にするとデータに合わせて自動調整されます
y_min = -0.5       # 最小値
y_max = 5.0        # 最大値
y_step = 0.5       # 目盛りの間隔
# ==========================================

output_dir_name = "fft_results_per_channel_nr500"
output_dir_path = os.path.join(os.getcwd(), output_dir_name)
os.makedirs(output_dir_path, exist_ok=True)

# 全ファイルを「末端フォルダ」ごとにグループ化
folder_to_files = defaultdict(list)
for file_path in target_files:
    folder_to_files[os.path.dirname(file_path)].append(file_path)

# 統合版グラフでチャンネルごとに色を変えるためのパレット
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]


def apply_axis_settings():
    """セル6・セル7で共通して使う軸の範囲・目盛り設定。"""
    plt.xlim(x_min, x_max)
    plt.xticks(np.arange(x_min, x_max + x_step, x_step))
    if y_max is not None:
        plt.ylim(y_min, y_max)
        if y_step is not None:
            plt.yticks(np.arange(y_min, y_max + y_step, y_step))
    plt.grid(True, linestyle="--", alpha=0.6)


print("✅ 準備完了。")
print(f"📁 出力先フォルダ: {output_dir_path}")
print(f"🎯 処理対象のチャンネル: {', '.join(target_channels)}")
print(f"📊 X軸設定: {x_min}〜{x_max}Hz (間隔:{x_step}Hz)")
if y_max is not None:
    print(f"📊 Y軸設定: {y_min}〜{y_max}MPa (間隔:{y_step}MPa)")
else:
    print("📊 Y軸設定: 自動調整")
print(f"📦 処理対象の末端フォルダ数: {len(folder_to_files)} 件")
print(f"📄 処理対象のファイル数: {len(target_files)} 件\n")
print("👉 次の「セル5」を実行してください。")

In [ ]:
# ==========================================================
# セル5：全ファイルのスペクトルを一括計算してメモリに保持する
# ==========================================================
# ここだけが重い処理です。ファイルは1回しか読みません。
# 計算済みスペクトルを spec_cache に持つので、セル6・セル7の描画は一瞬で終わり、
# 軸の設定を変えて描き直すときも再読み込みは不要です。
#
# 保持するのは x_min〜x_max に絞った点のみ（0〜150Hz なら1曲線751点）。
# 1290ファイル × 2ch でも約15MB に収まります。
# ==========================================================
print("🚀 スペクトルの一括計算を開始します...")
print(f"⏱️ 目安: 1ファイルあたり約0.6秒 → {len(target_files)} 件で約 "
      f"{len(target_files) * 0.6 / 60:.1f} 分\n")

spec_cache = {}   # file_path -> {channel: (freq, amp)}
errors = []
t0 = time.time()

for i, file_path in enumerate(target_files, 1):
    try:
        df, dt = read_nr500(file_path, channels=target_channels)
        entry = {}
        for ch in target_channels:
            y = df[ch].to_numpy(dtype=np.float64)
            entry[ch] = spectrum(y, dt, x_min, x_max)
        spec_cache[file_path] = entry
    except Exception as e:
        errors.append((file_path, f"{type(e).__name__}: {e}"))

    if i % 50 == 0 or i == len(target_files):
        el = time.time() - t0
        rest = el / i * (len(target_files) - i)
        print(f"  {i}/{len(target_files)} 件 完了  "
              f"（経過 {el:.0f}秒 / 残り約 {rest:.0f}秒）")

print("\n" + "-" * 40)
print(f"🏁 一括計算 完了！（成功: {len(spec_cache)} 件 / 失敗: {len(errors)} 件）")
if spec_cache:
    pts = sum(len(a) for e in spec_cache.values() for _, a in e.values())
    print(f"💾 キャッシュ保持点数: {pts:,} 点（約 {pts * 8 / 1e6:.1f} MB）")
if errors:
    print(f"⚠️ 失敗 {len(errors)} 件の詳細は「セル8」で確認できます。")
print("👉 次の「セル6」を実行してください。")

In [ ]:
# ==========================================================
# セル6：チャンネルごとの重ね描きグラフ（個別版）
# ==========================================================
# セル5のキャッシュから描くだけなので、ファイル再読み込みは発生せず一瞬で終わります。
# 軸設定を変えたい場合は「セル4の設定を書き換えてセル4→セル6」を実行してください
# （セル5の再実行は不要です）。
# ==========================================================
print("🚀 【個別版】チャンネルごとの重ね描きグラフ作成を開始します...")
success_individual = 0

for dirpath, files_in_dir in folder_to_files.items():
    tag = folder_tag(dirpath, main_dir)

    for ch in target_channels:
        plt.figure(figsize=(10, 5))
        drawn = 0

        for file_path in files_in_dir:
            entry = spec_cache.get(file_path)
            if entry is None or ch not in entry:
                continue
            freq, amp = entry[ch]
            plt.plot(freq, amp, alpha=0.5, linewidth=1.0)
            drawn += 1

        if drawn == 0:
            plt.close()
            continue

        plt.xlabel("Frequency [Hz]")
        plt.ylabel("Amplitude [MPa]")
        plt.title(f"FFT Spectrum - {tag} [{ch}] (Overlaid {drawn} files)")
        apply_axis_settings()
        plt.tight_layout()

        # 相対パスをタグに使うので 0.5mm/190℃ と 1.0mm/190℃ が衝突しない
        name = f"{tag}_{ch}_combined_fft.png"
        plt.savefig(os.path.join(output_dir_path, name), dpi=150)
        plt.close()

        success_individual += 1
        print(f"  --> ✅ 保存: {name}")

print("-" * 40)
print(f"🏁 個別グラフ完了！（生成: {success_individual} 枚）")
print("👉 次の「セル7」を実行してください。")

In [ ]:
# ==========================================================
# セル7：全チャンネルをまとめた統合グラフ（統合版）
# ==========================================================
# こちらもセル5のキャッシュから描くだけなので一瞬で終わります。
# ==========================================================
print("🚀 【統合版】全チャンネルのグラフ作成を開始します...")
success_integrated = 0

for dirpath, files_in_dir in folder_to_files.items():
    tag = folder_tag(dirpath, main_dir)
    parent_dir_path = os.path.dirname(dirpath)

    plt.figure(figsize=(12, 6))
    drawn_files = 0
    plotted_labels = set()

    for file_path in files_in_dir:
        entry = spec_cache.get(file_path)
        if entry is None:
            continue
        file_plotted = False

        for i, ch in enumerate(target_channels):
            if ch not in entry:
                continue
            freq, amp = entry[ch]
            # 凡例はチャンネルごとに1回だけ出す
            label = ch if ch not in plotted_labels else ""
            plt.plot(freq, amp, color=colors[i % len(colors)],
                     alpha=0.4, linewidth=1.0, label=label)
            if label:
                plotted_labels.add(ch)
            file_plotted = True

        if file_plotted:
            drawn_files += 1

    if drawn_files == 0:
        plt.close()
        continue

    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Amplitude [MPa]")
    plt.suptitle(f"Path: {parent_dir_path}", fontsize=10, color="dimgray", y=0.98)
    plt.title(f"FFT Spectrum - {tag} [ALL CHANNELS] (Overlaid {drawn_files} files)")
    apply_axis_settings()
    plt.legend(loc="upper right")
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    name = f"{tag}_ALL_combined_fft.png"
    plt.savefig(os.path.join(output_dir_path, name), dpi=150)
    success_integrated += 1
    print(f"  --> 🌟 保存: {name}")

    plt.show()   # ノートブック上にも表示

print("-" * 40)
print(f"🏁 統合グラフ完了！（生成: {success_integrated} 枚）")
print(f"📁 出力先: {output_dir_path}")
print("👉 失敗ファイルの詳細を見る場合は「セル8」を実行してください。")

In [ ]:
# ==========================================================
# セル8：読み込みに失敗したファイルの詳細レポート
# ==========================================================
if "errors" not in globals():
    print("⚠️ セル5がまだ実行されていません。先にセル5を実行してください。")

elif not errors:
    print("✅ 読み込みに失敗したファイルはありませんでした。")
    print(f"   対象 {len(target_files)} 件すべてを正常に処理しています。")

else:
    print(f"🚨 {len(errors)} 件のファイルで読み込みに失敗しました:\n")
    for path, msg in errors:
        print(f" ❌ {os.path.basename(path)}")
        print(f"      場所: {os.path.dirname(path)}")
        print(f"      原因: {msg}")
        print()

    print("-" * 50)
    print("【よくある原因】")
    print(" ・ファイルが空、または途中で切れている")
    print(" ・ヘッダの「データ数」と実際の行数が食い違っている")
    print(" ・指定したチャンネル（target_channels）がそのファイルに存在しない")
    print("上記のファイルを Excel 等で直接開いて中身を確認してください。")